In [3]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from scipy.stats import brunnermunzel
from scipy.stats import gaussian_kde

# LOAD DATA


df = pd.read_excel(
    r"C:\Users\Seun.Ale\OneDrive - Technological University Dublin\Desktop\Fixed vs Dynamic Partnership\..Latest M4.tmp stable vs dynamic experiment-table.xlsx"
)

# CLEAN


df = df.dropna(
    subset=["proximity-risk?"]
)

dynamic = df[
    df["proximity-risk?"]==1
]

fixed = df[
    df["proximity-risk?"]==0
]


# MODEL OUTCOMES

OUTCOMES = {

    "Incidence":
        "Inc_Y5",

    "Prevalence":
        "chronic_Y5",

    "Treatment":
        "Treat_Y5",

    "Cures":
        "Cured_Y5"

}


# SUPERIORITY EXPERIMENT


def superiority(a,b):

    g=0
    e=0

    for x in a:

        for y in b:

            if x>y:

                g+=1

            elif x==y:

                e+=1

    return (
        g+0.5*e
    )/(len(a)*len(b))

# SUMMARY TABLE


summary=[]


# PLOT FUNCTION

def make_distribution_plot(
        dynamic_vals,
        fixed_vals,
        outcome_name):

    dyn_med=np.median(dynamic_vals)

    fix_med=np.median(fixed_vals)

    dyn_iqr=(
        np.percentile(dynamic_vals,75)
        -
        np.percentile(dynamic_vals,25)
    )

    fix_iqr=(
        np.percentile(fixed_vals,75)
        -
        np.percentile(fixed_vals,25)
    )

    xmin=min(
        dynamic_vals.min(),
        fixed_vals.min()
    )

    xmax=max(
        dynamic_vals.max(),
        fixed_vals.max()
    )

    x=np.linspace(
        xmin,
        xmax,
        500
    )

    fig,ax=plt.subplots(
        1,
        2,
        figsize=(12,5),
        dpi=300
    )



    kde=gaussian_kde(
        dynamic_vals
    )

    ax[0].hist(
        dynamic_vals,
        bins=20,
        density=True,
        alpha=.6
    )

    ax[0].plot(
        x,
        kde(x),
        linewidth=2
    )

    ax[0].axvline(
        dyn_med,
        linestyle="--"
    )

    ax[0].set_title(
        "Dynamic Partnership"
    )

    ax[0].legend([
        f"Median={dyn_med:.1f}\nIQR={dyn_iqr:.2f}"
    ])

    kde=gaussian_kde(
        fixed_vals
    )

    ax[1].hist(
        fixed_vals,
        bins=20,
        density=True,
        alpha=.6
    )

    ax[1].plot(
        x,
        kde(x),
        linewidth=2
    )

    ax[1].axvline(
        fix_med,
        linestyle="--"
    )

    ax[1].set_title(
        "Fixed Partnership"
    )

    ax[1].legend([
        f"Median={fix_med:.1f}\nIQR={fix_iqr:.2f}"
    ])

    plt.suptitle(
        outcome_name
    )

    plt.tight_layout()

    plt.savefig(
        f"{outcome_name}.png"
    )

    plt.close()


# OUTCOME ANALYSIS

for name,col in OUTCOMES.items():

    dyn=dynamic[
        col
    ].dropna().values

    fix=fixed[
        col
    ].dropna().values

    bm,p=brunnermunzel(
        fix,
        dyn
    )

    sup=superiority(
        fix,
        dyn
    )

    summary.append({

        "Outcome":
            name,

        "Dynamic median":
            np.median(dyn),

        "Dynamic IQR":
            np.percentile(dyn,75)
            -
            np.percentile(dyn,25),

        "Fixed median":
            np.median(fix),

        "Fixed IQR":
            np.percentile(fix,75)
            -
            np.percentile(fix,25),

        "BM":
            bm,

        "p":
            p,

        "Superiority":
            sup

    })

    make_distribution_plot(
        dyn,
        fix,
        name
    )


# MRE ANALYSIS


real=np.array(
    [698,601,552,538,511]
)

def mre(sim):

    return np.mean(
        np.abs(
            sim-real
        )/real
    )

dyn_matrix=dynamic[
[
"chronic_Y1",
"chronic_Y2",
"chronic_Y3",
"chronic_Y4",
"chronic_Y5"
]
].values

fix_matrix=fixed[
[
"chronic_Y1",
"chronic_Y2",
"chronic_Y3",
"chronic_Y4",
"chronic_Y5"
]
].values

dyn_mre=np.array([
    mre(x)
    for x in dyn_matrix
])

fix_mre=np.array([
    mre(x)
    for x in fix_matrix
])

bm,p=brunnermunzel(
    fix_mre,
    dyn_mre
)

summary.append({

"Outcome":"MRE",

"Dynamic median":
np.median(dyn_mre),

"Dynamic IQR":
np.percentile(
dyn_mre,
75
)-np.percentile(
dyn_mre,
25
),

"Fixed median":
np.median(fix_mre),

"Fixed IQR":
np.percentile(
fix_mre,
75
)-np.percentile(
fix_mre,
25
),

"BM":bm,

"p":p,

"Superiority":
superiority(
fix_mre,
dyn_mre
)

})

make_distribution_plot(
dyn_mre,
fix_mre,
"MRE"
)


# EXPORT TABLE


summary=pd.DataFrame(
summary
)

summary.to_csv(
"manuscript_summary.csv",
index=False
)

print(summary)

      Outcome  Dynamic median  Dynamic IQR  Fixed median  Fixed IQR        BM  \
0   Incidence        7.000000     3.000000      9.000000   4.000000 -7.536608   
1  Prevalence      544.000000    18.500000    555.500000  23.000000 -5.293197   
2   Treatment       18.000000     5.250000     18.000000   6.250000 -1.283968   
3       Cures      455.500000    19.250000    462.500000  22.000000 -3.694543   
4         MRE        0.217899     0.025612      0.237697   0.028358 -5.992516   

              p  Superiority  
0  1.689466e-12      0.75430  
1  3.193691e-07      0.69610  
2  2.006544e-01      0.55225  
3  2.856690e-04      0.64345  
4  9.610077e-09      0.71840  
